# Turb-DETR Week 2A: Data Prep + SimAM Injection
**NO GPU needed.** Estimated: ~30-40 min

Does: augment training images, create combined dataset, patch Ultralytics with SimAM, save to Drive.

In [ ]:
import os, time
t0 = time.time()
from google.colab import drive
drive.mount('/content/drive')

DATASET = "/content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset"
if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for n in os.listdir(base):
        if "icra" in n.lower():
            for sub in ["dataset", "Dataset", ""]:
                cand = os.path.join(base, n, sub) if sub else os.path.join(base, n)
                if os.path.exists(os.path.join(cand, "train")):
                    DATASET = cand; break
assert os.path.exists(os.path.join(DATASET, "train"))
os.makedirs("/content/turb-detr", exist_ok=True)
os.chdir("/content/turb-detr")
print(f"Dataset: {DATASET} ({time.time()-t0:.1f}s)")


In [ ]:
import os, subprocess, time
from pathlib import Path
t0 = time.time()
SRC = Path(DATASET)
OUT = Path("/content/turb-detr/data/processed")
LOCAL = Path("/content/local_labels")

for split in ["train", "val", "test"]:
    src_dir = SRC / split
    img_out = OUT / split / "images"
    img_out.mkdir(parents=True, exist_ok=True)
    for f in src_dir.glob("*.jpg"):
        dst = img_out / f.name
        if not dst.exists(): os.symlink(f.resolve(), dst)

    local_lbl = LOCAL / split
    local_lbl.mkdir(parents=True, exist_ok=True)
    subprocess.run(f"cp {src_dir}/*.txt {local_lbl}/ 2>/dev/null", shell=True)

    lbl_out = OUT / split / "labels"
    lbl_out.mkdir(parents=True, exist_ok=True)
    rov = 0
    for txt in local_lbl.glob("*.txt"):
        lines = []
        for line in open(txt):
            parts = line.strip().split()
            if len(parts) == 5 and int(parts[0]) != 2:
                lines.append(line.strip())
            elif len(parts) == 5 and int(parts[0]) == 2:
                rov += 1
        with open(lbl_out / txt.name, "w") as f:
            f.write("\n".join(lines) + "\n" if lines else "")
    print(f"  {split}: {rov} ROV removed")

bad = OUT / "train" / "images" / "obj1658_frame0002383 (1).jpg"
if bad.exists() or bad.is_symlink(): os.unlink(bad)
print(f"Done ({time.time()-t0:.1f}s)")


In [ ]:
import cv2, numpy as np, shutil, time
from pathlib import Path
from tqdm import tqdm
t0 = time.time()

def jaffe(image, c=0.15, depth=3.0, seed=None):
    rng = np.random.RandomState(seed)
    img = image.astype(np.float32) / 255.0
    att = np.exp(-c * depth)
    bs = np.zeros_like(img)
    bs[:,:,0] = 0.10 * (1 - att)
    bs[:,:,1] = 0.30 * (1 - att)
    bs[:,:,2] = 0.05 * (1 - att)
    t = img * att + bs
    n = rng.normal(0, 0.02 * c * 10, img.shape).astype(np.float32)
    return (np.clip(t + n, 0, 1) * 255).astype(np.uint8)

LEVELS = {"light": 0.05, "medium": 0.15, "heavy": 0.30}
TRAIN_IMG = Path("data/processed/train/images")
TRAIN_LBL = Path("data/processed/train/labels")
TEST_IMG = Path("data/processed/test/images")
TEST_LBL = Path("data/processed/test/labels")

# Augment TRAINING
train_imgs = sorted(TRAIN_IMG.glob("*.jpg"))
print(f"Augmenting {len(train_imgs)} train images x 3 levels...")
for lname, cval in LEVELS.items():
    oimg = Path(f"data/augmented_train/{lname}/images"); oimg.mkdir(parents=True, exist_ok=True)
    olbl = Path(f"data/augmented_train/{lname}/labels"); olbl.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(train_imgs, desc=f"train-{lname}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oimg / f"{lname}_{ip.name}"), jaffe(img, c=cval, seed=42+i))
        ls = TRAIN_LBL / f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, olbl / f"{lname}_{ip.stem}.txt")

# Augment TEST
test_imgs = sorted(TEST_IMG.glob("*.jpg"))
print(f"\nAugmenting {len(test_imgs)} test images x 3 levels...")
for lname, cval in LEVELS.items():
    oimg = Path(f"data/augmented_test/{lname}/images"); oimg.mkdir(parents=True, exist_ok=True)
    olbl = Path(f"data/augmented_test/{lname}/labels"); olbl.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(test_imgs, desc=f"test-{lname}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oimg / ip.name), jaffe(img, c=cval, seed=42+i))
        ls = TEST_LBL / f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, olbl / ls.name)

print(f"\nDone ({time.time()-t0:.1f}s)")


In [ ]:
import os
from pathlib import Path

CIMG = Path("data/combined_train/images"); CIMG.mkdir(parents=True, exist_ok=True)
CLBL = Path("data/combined_train/labels"); CLBL.mkdir(parents=True, exist_ok=True)

# Clean
c = 0
for f in Path("data/processed/train/images").glob("*.jpg"):
    d = CIMG / f.name
    if not d.exists(): os.symlink(f.resolve(), d); c += 1
for f in Path("data/processed/train/labels").glob("*.txt"):
    d = CLBL / f.name
    if not d.exists(): os.symlink(f.resolve(), d)

# Augmented
for level in ["light", "medium", "heavy"]:
    for f in Path(f"data/augmented_train/{level}/images").glob("*.jpg"):
        d = CIMG / f.name
        if not d.exists(): os.symlink(f.resolve(), d)
    for f in Path(f"data/augmented_train/{level}/labels").glob("*.txt"):
        d = CLBL / f.name
        if not d.exists(): os.symlink(f.resolve(), d)

total = len(list(CIMG.glob("*.jpg")))
print(f"Combined: {total} images ({total/c:.1f}x clean)")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import yaml, os
from pathlib import Path
os.makedirs("configs/datasets", exist_ok=True)

with open("configs/datasets/trash_icra19_clean.yaml", "w") as f:
    yaml.dump({"path": str(Path("data/processed").resolve()), "train": "train/images",
               "val": "val/images", "test": "test/images", "names": {0: "plastic", 1: "bio"}}, f)

with open("configs/datasets/trash_icra19_augmented.yaml", "w") as f:
    yaml.dump({"path": str(Path("data").resolve()), "train": "combined_train/images",
               "val": "processed/val/images", "test": "processed/test/images",
               "names": {0: "plastic", 1: "bio"}}, f)

for level in ["light", "medium", "heavy"]:
    with open(f"configs/datasets/trash_icra19_turbid_{level}.yaml", "w") as f:
        yaml.dump({"path": str(Path(f"data/augmented_test/{level}").resolve()),
                   "train": "images", "val": "images", "test": "images",
                   "names": {0: "plastic", 1: "bio"}}, f)

print("Configs created:")
for f in sorted(Path("configs/datasets").glob("*.yaml")): print(f"  {f.name}")


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"], check=True)

import ultralytics
from pathlib import Path

ul_path = Path(ultralytics.__file__).parent
block_py = ul_path / "nn" / "modules" / "block.py"
source = block_py.read_text()

if "class SimAM" in source:
    print("SimAM already injected")
else:
    # 1. SimAM class
    simam_class = """
# === TURB-DETR SimAM (Yang et al. 2021) ===
class SimAM(nn.Module):
    def __init__(self, lam=1e-4):
        super().__init__()
        self.lam = lam
    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        mu = x.mean(dim=[2, 3], keepdim=True)
        var = ((x - mu) ** 2).sum(dim=[2, 3], keepdim=True) / n
        e = (x - mu) ** 2 / (4 * (var + self.lam)) + 0.5
        return x * torch.sigmoid(e)
# === END SimAM ===
"""
    # Insert after imports
    lines = source.split("\n")
    idx = 0
    for i, line in enumerate(lines):
        if line.startswith("import ") or line.startswith("from "):
            idx = i + 1
        if line.startswith("class "):
            break
    lines.insert(idx, simam_class)
    source = "\n".join(lines)

    # 2. Add self.simam to HybridEncoder.__init__
    source = source.replace(
        "self.input_proj = nn.ModuleList()",
        "self.input_proj = nn.ModuleList()\n        self.simam = SimAM()"
    )

    # 3. Apply in forward
    source = source.replace(
        "return outs",
        "outs = [self.simam(o) for o in outs]  # SimAM\n        return outs",
        1
    )

    block_py.write_text(source)
    print("SimAM injected")

# Verify
s = block_py.read_text()
assert "class SimAM" in s, "SimAM class missing"
assert "self.simam" in s, "self.simam missing"
print("All injection points verified")


In [ ]:
import importlib, ultralytics
importlib.reload(ultralytics)
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")
params = sum(p.numel() for p in model.model.parameters())
print(f"Parameters: {params:,}")
print(f"Expected ~32,810,186 (vanilla RT-DETR-L)")
diff = abs(params - 32810186)
if diff < 100:
    print("ZERO parameter increase confirmed")
else:
    print(f"WARNING: diff = {diff}")


In [ ]:
import shutil
from pathlib import Path
import ultralytics

SAVE = Path("/content/drive/MyDrive/turb_detr_results/week2_prep")
SAVE.mkdir(parents=True, exist_ok=True)

block_py = Path(ultralytics.__file__).parent / "nn" / "modules" / "block.py"
shutil.copy2(block_py, SAVE / "block_patched.py")
for f in Path("configs/datasets").glob("*.yaml"):
    shutil.copy2(f, SAVE / f.name)

print(f"Saved to: {SAVE}")
print("Now run Week 2B training notebook (GPU required)")
